# Notebook 06: FastAPI Inference Service

## English
This notebook validates a minimal FastAPI service for FinBERT inference, including local server startup and test requests.

**Goals:**
- Start a local API for sentiment prediction.
- Confirm documentation and endpoint behavior.
- Log example requests for reproducibility.

## Espanol
Este notebook valida un servicio FastAPI minimo para inferencia con FinBERT.

**Objetivos:**
- Levantar un API local para prediccion de sentimiento.
- Confirmar la documentacion y comportamiento del endpoint.
- Registrar requests de ejemplo para reproducibilidad.

In [1]:
!pip install fastapi uvicorn pydantic nest-asyncio

  Using cached uvicorn-0.40.0-py3-none-any.whl.metadata (6.7 kB)
  Using cached starlette-0.52.1-py3-none-any.whl.metadata (6.3 kB)
Using cached starlette-0.52.1-py3-none-any.whl (74 kB)
Using cached uvicorn-0.40.0-py3-none-any.whl (68 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [fastapi]


In [ ]:
import torch
from transformers import pipeline
import nest_asyncio
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import uvicorn

# Permitir que uvicorn corra dentro de un notebook
nest_asyncio.apply()

# Configuración del modelo
MODEL_PATH = "../models/finbert_final"
device = 0 if torch.cuda.is_available() else -1

print(f"Cargando modelo desde {MODEL_PATH} en {'GPU' if device==0 else 'CPU'}...")
nlp_pipe = pipeline(
    "text-classification", model=MODEL_PATH, tokenizer=MODEL_PATH, device=device
)


# Definir el esquema de entrada
class SentimentRequest(BaseModel):
    text: str


# Definir el esquema de salida
class SentimentResponse(BaseModel):
    text: str
    label: str
    score: float

Cargando modelo desde ../models/finbert_final en GPU...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
app = FastAPI(
    title="Financial Sentiment Analysis API",
    description="API para clasificar sentimiento financiero usando FinBERT Fine-tuned",
    version="1.0.0",
)


@app.get("/")
def home():
    return {
        "message": "Financial Sentiment API is running. Go to /docs for Swagger UI."
    }


@app.post("/predict", response_model=SentimentResponse)
def predict_sentiment(request: SentimentRequest):
    if not request.text.strip():
        raise HTTPException(status_code=400, detail="El texto no puede estar vacío")

    try:
        # Realizar predicción
        result = nlp_pipe(request.text, truncation=True, max_length=512)[0]

        return SentimentResponse(
            text=request.text, label=result["label"], score=round(result["score"], 4)
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

In [ ]:
import uvicorn
import nest_asyncio
from threading import Thread

nest_asyncio.apply()


def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="info")


# Lanzar el servidor en un hilo aparte
server_thread = Thread(target=run_server, daemon=True)
server_thread.start()

print("✅ Servidor iniciado en http://127.0.0.1:8000")
print("➡ Abre http://127.0.0.1:8000/docs para probar la API desde Swagger UI")

✅ Servidor iniciado en http://127.0.0.1:8000
➡ Abre http://127.0.0.1:8000/docs para probar la API desde Swagger UI


INFO:     Started server process [64947]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:55246 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:55246 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:34688 - "POST /predict HTTP/1.1" 422 Unprocessable Entity
INFO:     127.0.0.1:43078 - "POST /predict HTTP/1.1" 422 Unprocessable Entity
INFO:     127.0.0.1:46308 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:45098 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:45562 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:59472 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:59478 - "POST /predict HTTP/1.1" 200 OK


# Results Snapshot

## English
- Local server starts at http://127.0.0.1:8000 and docs are reachable at /docs.
- Sample requests return valid predictions and are logged in the notebook output.
- The service exposes a clean interface for dashboard or batch integration.

## Espanol
- El servidor local inicia en http://127.0.0.1:8000 y /docs esta disponible.
- Los requests de prueba devuelven predicciones validas y quedan registrados.
- El servicio expone una interfaz limpia para integracion con dashboard o batch.

# Conclusions

## English
- FastAPI provides a lightweight path to productionize FinBERT inference.
- Add authentication and monitoring before deploying to production.
- Containerization is recommended for consistent deployments.

## Espanol
- FastAPI ofrece un camino ligero para llevar FinBERT a produccion.
- Agregar autenticacion y monitoreo antes de desplegar.
- Se recomienda containerizar para despliegues consistentes.